# Monte Carlo Simulations

In this notebook, I run Monte Carlo simulations and view the results of varying alpha/beta/gamma.

In [1]:
# Libraries

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
os.getcwd()

"/Users/jananidhileepan/Desktop/Don't. Even./University/Imperial College London/Year 2/Dissertation/GitHub/fairness_smoke_alarm/src"

In [ ]:
from changepoint_detection_functions import (
    CUSUMDetector,
    referee_cusum,
    EWMAResidualDetector,
    referee_ewma,
    and_or_gate,
    and_or_gate_referee,
    miss_rate
)

from fairness_metrics import unfairness_score

ImportError: cannot import name 'and_or_gate_referee' from 'changepoint_detection_functions' (/Users/jananidhileepan/Desktop/Don't. Even./University/Imperial College London/Year 2/Dissertation/GitHub/fairness_smoke_alarm/src/changepoint_detection_functions.py)

## 1. Convert Boostrapped Standard Errors to a 3x3 Matrix

In [9]:
boot_df = pd.read_csv("../data/processed/bootstrap.csv", index_col = 0)

Sigma = np.cov(boot_df[["DP", "EO", "CAL"]].values, rowvar=False)

In [10]:
Sigma

array([[ 1.99191513e-06, -1.09625907e-07, -5.68874996e-08],
       [-1.09625907e-07,  4.07732412e-07, -2.76369006e-08],
       [-5.68874996e-08, -2.76369006e-08,  1.64911347e-07]])

In [13]:
def sigma_hat(w, Sigma):
    return np.sqrt(w @ Sigma @ w)

w = np.array([1/3, 1/3, 1/3])
sigma_hat(w, Sigma)

0.0004917382180160006

In [14]:
def evaluate_weighting(alpha, beta, gamma, df_fairness, Sigma,
                       referee_cusum_alarm, referee_ewma_alarm,
                       referee_and, referee_or,
                       k=0.5, h=4.77, baseline_year=2007):

    composite = unfairness_score(df_fairness, alpha, beta, gamma)   # weights passed
    w = np.array([alpha, beta, gamma])                              # order: DP, EO, CAL
    se = sigma_hat(w, Sigma)

    cus = CUSUMDetector(composite, sigma_hat=se, k=k, h=h, baseline_year=baseline_year)
    cus.run()

    ewm = EWMAResidualDetector(composite, sigma_hat=se, baseline_year=baseline_year)
    ewm.run()

    and_alarm = and_or_gate(cus, ewm, gate="and")
    or_alarm  = and_or_gate(cus, ewm, gate="or")

    weights = {"alpha": alpha, "beta": beta, "gamma": gamma, "sigma_hat": se}

    return [
        miss_rate(cus.alarm,  referee_cusum_alarm, variant="cusum", **weights),
        miss_rate(ewm.alarm,  referee_ewma_alarm,  variant="ewma",  **weights),
        miss_rate(and_alarm,  referee_and,         variant="and",   **weights),
        miss_rate(or_alarm,   referee_or,          variant="or",    **weights),
    ]